# Глава 3 — Reasoning

Глава не меняет цикл TinyAgent. Здесь показаны prompting, self-consistency, Best-of-N и native reasoning на локальном LLM.

В книге prompting демонстрируется на `gemma3:12b`. По умолчанию здесь используется установленная `gemma4:e4b` с отключённым native reasoning. Для точного повторения сравнения выберите `gemma3:12b`, если она уже скачана. Результаты вероятностные: правильный ответ не гарантирован.

In [ ]:
import json
import re
from collections import Counter

from llm import LLM

MODEL = "gemma4:e4b"
SAMPLES = 3  # В книге 10; увеличьте для более широкого поиска.
llm = LLM(model=MODEL, think=False, temperature=1)

## Прямой ответ и Chain-of-Thought

Промпт меняет поведение генерации, не веса модели.

In [ ]:
query = "There were 9 penguins in view. 2 disappeared into the water and 4 more arrived. How many penguins are visible now?"
print(llm.generate([{"role": "user", "content": query + " Give only the number."}]).content)

In [ ]:
messages = [
    {"role": "user", "content": "There are 8 birds. 3 fly away. How many remain?"},
    {"role": "assistant", "content": "Start with 8 birds and subtract 3: 8 - 3 = 5. Answer: 5"},
    {"role": "user", "content": query},
]
print(llm.generate(messages).content)

In [ ]:
reasoning_prompt = query + " Solve step by step. Finish with Answer: <integer>."
print(llm.generate([{"role": "user", "content": reasoning_prompt}]).content)

## Self-consistency

Выбираем самый частый итоговый ответ. Отдельно учитываем ответы, которые не удалось разобрать. При равенстве голосов показываем всех лидеров.

In [ ]:
def extract_answer(content):
    matches = re.findall(r"Answer:\s*(-?\d+)\s*\.?\s*$", (content or "").replace("**", ""), flags=re.IGNORECASE)
    return int(matches[-1]) if matches else None

responses = [llm.generate([{"role": "user", "content": reasoning_prompt}]) for _ in range(SAMPLES)]
answers = [extract_answer(response.content) for response in responses]
counts = Counter(answer for answer in answers if answer is not None)
leaders = [answer for answer, count in counts.items() if count == max(counts.values())] if counts else []
print("Votes:", counts)
print("Unparsed:", answers.count(None))
print("Leading answers:", leaders)

## Best-of-N с проверяемым результатом

Тот же принцип, что в книге: генерируем варианты и оцениваем проверкой. Здесь проверяем JSON с переводом римских чисел, чтобы пример не требовал выполнения сгенерированного Python-кода. Эта проверка измеряет ответы на фиксированные примеры, а не качество универсального конвертера.

In [ ]:
test_cases = {"III": 3, "IV": 4, "IX": 9, "LVIII": 58, "MCMXCIV": 1994, "IIII": 0, "IC": 0, "ABC": 0}

def verify(content):
    try:
        result = json.loads(content or "")
    except json.JSONDecodeError:
        return 0.0
    if not isinstance(result, dict):
        return 0.0
    return sum(type(result.get(key)) is int and result[key] == expected for key, expected in test_cases.items()) / len(test_cases)

prompt = (
    "Convert each Roman numeral into an integer. Use 0 for invalid numerals. "
    "Return only a JSON object mapping each input to an integer, without markdown: "
    + json.dumps(list(test_cases))
)
candidates = [llm.generate([{"role": "user", "content": prompt}]) for _ in range(SAMPLES)]
scored = [(response.content, verify(response.content)) for response in candidates]
best_answer, best_score = max(scored, key=lambda item: item[1])
print("Scores:", [score for _, score in scored])
print("Best score:", best_score)
print("Best answer:", best_answer)

## Native reasoning

Для этой ячейки нужна модель с поддержкой native reasoning. `think=True` разрешает стандартное поведение backend; формат поля reasoning зависит от сервера и модели.

In [ ]:
native_llm = LLM(model="gemma4:e4b", think=True)
response = native_llm.generate([{"role": "user", "content": query}])
print("Reasoning:", response.reasoning)
print("Answer:", response.content)